# 4.12 Train-Test Split

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 4: Data Preprocessing & Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-04-data-preprocessing-and-exploratory-data-analysis/04.12_train-test-split.ipynb)

Companion code for Section 4.12 of the book. Run the cells in order from top to bottom.

**Example 4.12.1**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X, y = data.data, data.target

# ── Basic split ───────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,    # 20% for testing
    random_state=42,   # Reproducibility - ALWAYS set this
    shuffle=True       # Shuffle before splitting (default=True)
)

print(f'Total samples:  {len(X):,}')
print(f'Training:       {len(X_train):,} ({len(X_train)/len(X):.0%})')
print(f'Test:           {len(X_test):,} ({len(X_test)/len(X):.0%})')

# ── Stratified split - critical for imbalanced datasets ───────────
# Ensures each split has the same class distribution as the full dataset
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('\nClass distribution after stratified split:')
for name, y_split in [('Full', y), ('Train', y_tr_s), ('Test', y_te_s)]:
    print(f'  {name:6s}: class 0 = {(y_split==0).mean():.1%}, class 1 = {(y_split==1).mean():.1%}')

# ── Three-way split: train / validation / test ────────────────────
# Train: model training
# Validation: hyperparameter tuning and model selection
# Test: FINAL unbiased evaluation - touch only once!
X_temp, X_test3, y_temp, y_test3 = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)
X_train3, X_val3, y_train3, y_val3 = train_test_split(
    X_temp, y_temp, test_size=0.146, stratify=y_temp, random_state=42
    # 0.146 of remaining 85% ≈ 15% of total
)

print(f'\n3-way split (target: 70/15/15):')
print(f'  Train:      {len(X_train3):,} ({len(X_train3)/len(X):.0%})')
print(f'  Validation: {len(X_val3):,}  ({len(X_val3)/len(X):.0%})')
print(f'  Test:       {len(X_test3):,}  ({len(X_test3)/len(X):.0%})')

# ── DATA LEAKAGE WARNING ──────────────────────────────────────────
print('\n=== DATA LEAKAGE RULES ===')
print('NEVER fit any preprocessor (scaler, imputer, encoder) on the test set.')
print('NEVER use test set information in feature engineering.')
print('NEVER tune hyperparameters using test set performance.')
print('ALWAYS fit preprocessors on train set, transform test set.')

# Correct preprocessing pipeline
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # FIT on train only
X_test_scaled  = scaler.transform(X_test)         # TRANSFORM test (no fit!)

**Example 4.12.2**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, GroupShuffleSplit

# ── Time-series split - respect temporal ordering ─────────────────
# NEVER random-shuffle time series data - future data would leak into training!
np.random.seed(42)
n_days = 365
ts_df = pd.DataFrame({
    'date':    pd.date_range('2023-01-01', periods=n_days),
    'sales':   100 + np.cumsum(np.random.randn(n_days)) + np.sin(np.arange(n_days)*2*np.pi/7)*20,
})

# Correct: sort by time, take last N as test
cutoff_date = '2023-10-01'
train_ts = ts_df[ts_df.date < cutoff_date]
test_ts  = ts_df[ts_df.date >= cutoff_date]
print(f'Time-series split:')
print(f'  Train: {train_ts.date.min().date()} to {train_ts.date.max().date()} ({len(train_ts)} days)')
print(f'  Test:  {test_ts.date.min().date()} to {test_ts.date.max().date()} ({len(test_ts)} days)')

# ── Group split - patients, users, stores must not span train/test ─
# If patient 42's data is in both train and test, the model learns
# patient-specific patterns that don't generalise to new patients
patient_ids = np.repeat(np.arange(50), 20)    # 50 patients, 20 samples each
X_patients  = np.random.randn(1000, 10)
y_patients  = np.random.choice([0,1], 1000)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_patients, y_patients, groups=patient_ids))

print(f'\nGroup split (by patient):')
print(f'  Train patients: {np.unique(patient_ids[train_idx]).shape[0]}')
print(f'  Test patients:  {np.unique(patient_ids[test_idx]).shape[0]}')
print(f'  Overlap:        {len(set(patient_ids[train_idx]) & set(patient_ids[test_idx]))}')
print(f'  (Zero overlap = correct - no patient appears in both sets)')